## Metadata Query Agent — AgentCore Online Evaluation

This notebook sets up **online (continuous) evaluation** of the Metadata Query Agent deployed on **Amazon Bedrock AgentCore Runtime**. Unlike [notebook 3](./3_metadata_query_agent_ac_eval-on-demand.ipynb) which runs on-demand evaluation against selected sessions, online evaluation runs automatically against all production traffic based on a sampling rate.

### What this notebook does
1. **Restores** agent context from notebook 3 (`ondemand_agent_id` via `%store`)
2. **Looks up** the AgentCore Runtime ARN from CloudFormation stack outputs
3. **Creates** an online evaluation configuration with four built-in evaluators
4. **Invokes** the agent with test queries to generate new traces for evaluation
5. **Confirms** the online evaluation config is active and evaluating live traffic

### On-Demand vs Online Evaluation

| Aspect | On-Demand (notebook 3) | Online (this notebook) |
|:-------|:-----------------------|:-----------------------|
| Trigger | Developer-initiated | Continuous / automatic |
| Scope | Selected sessions / traces | All production traffic (sampled) |
| Use case | Targeted investigation, regression testing | Production monitoring |
| Results | Immediate in notebook | AgentCore Observability dashboard |

### Prerequisites

- **Notebook 1** (`1_metadata_agent_strands_eval.ipynb`) must have run to `%store metadata_id`
- **Notebook 3** (`3_metadata_query_agent_ac_eval-on-demand.ipynb`) must have run to `%store ondemand_agent_id`
- **AgentCore stack** must be deployed (`cdk deploy semantic-layer-agentcore`)
- **Metadata Query Agent** must be running on AgentCore Runtime

In [1]:
# Prereq:
# !pip install bedrock-agentcore-starter-toolkit==0.3.3 python-dotenv pandas --quiet

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac-demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
%store -r metadata_id
%store -r ondemand_agent_id

try:
    EVAL_ID = metadata_id
    agent_id = ondemand_agent_id
    print(f"✓ Loaded from %store:")
    print(f"  metadata_id        = {EVAL_ID}")
    print(f"  ondemand_agent_id  = {agent_id}")
except NameError as e:
    raise Exception(
        "Missing stored values. Please run notebook 1 (metadata_id) and "
        "notebook 3 (ondemand_agent_id) before executing this notebook."
    ) from e

# ── Look up AgentCore Runtime ARN and CDK eval execution role from CloudFormation ──
cfn = session.client('cloudformation', region_name=region)

stack_name = f'{PROJECT_NAME}-agentcore'
outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}
METADATA_QUERY_RUNTIME_ARN = output_map['MetadataQueryRuntimeArn']

eval_stack_name = f'{PROJECT_NAME}-agentcore-eval'
eval_outputs = cfn.describe_stacks(StackName=eval_stack_name)['Stacks'][0]['Outputs']
eval_output_map = {o['OutputKey']: o['OutputValue'] for o in eval_outputs}
EVAL_EXECUTION_ROLE_ARN = eval_output_map['EvalExecutionRoleArn']

print(f"\n✓ AgentCore Runtime details (from stack '{stack_name}'):")
print(f"  Runtime ARN:          {METADATA_QUERY_RUNTIME_ARN}")
print(f"  Agent ID:             {agent_id}")
print(f"  Eval Execution Role:  {EVAL_EXECUTION_ROLE_ARN}")

# ── Load evaluation dataset ──
with open('../data/eval/metadata_query_agent_evaluation_dataset.json', 'r') as f:
    test_cases = json.load(f)

print(f"\n✓ Loaded {len(test_cases)} test case(s):")
for tc in test_cases:
    print(f"  [{tc['id']}] ({tc['category']}): {tc['query']}")

✓ Loaded from %store:
  metadata_id        = semantic-rag-single_table_basic-8f7bcf77
  ondemand_agent_id  = semantic_layer_metadata_query-itnnG8AGV6

✓ AgentCore Runtime details (from stack 'semantic-layer-agentcore'):
  Runtime ARN:          arn:aws:bedrock-agentcore:us-east-1:381492284087:runtime/semantic_layer_metadata_query-itnnG8AGV6
  Agent ID:             semantic_layer_metadata_query-itnnG8AGV6
  Eval Execution Role:  arn:aws:iam::381492284087:role/semantic-layer-agentcore--EvalExecutionRole59F08324-ZX0EbSgXy1pm

✓ Loaded 1 test case(s):
  [test1] (admin_codes_test1): How many unique admincodes exist?


### Initialize AgentCore Evaluation Client

In [4]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=region)
print(f"✓ AgentCore Evaluation client initialized (region={region})")

/Users/huthmac/Documents/AWS/00_workspace/semantic-layer/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✓ AgentCore Evaluation client initialized (region=us-east-1)


### Create Online Evaluation Configuration

We configure online evaluation with four built-in evaluators at different granularity levels:

| Evaluator | Level | What it measures |
|:----------|:------|:-----------------|
| `Builtin.GoalSuccessRate` | Session | Did the agent achieve the user's data question end-to-end? |
| `Builtin.Correctness` | Trace | Is the agent's SQL answer factually correct? |
| `Builtin.ToolSelectionAccuracy` | Span | Did the agent call `retrieve_kb_context` → `disambiguate` → `execute_sql` in order? |
| `Builtin.ToolParameterAccuracy` | Span | Were SQL query, `database_name`, and `catalog_id` parameters correct? |

We use `sampling_rate=100` for this evaluation environment. In production, set this lower (e.g. `10` for 10%) to reduce costs.

In [5]:
# API constraint: onlineEvaluationConfigName must match [a-zA-Z][a-zA-Z0-9_]{0,47}
# (no hyphens, max 48 chars) — replace hyphens with underscores
online_config_name = f"{PROJECT_NAME.replace('-', '_')}_metadata_query_online_eval"

# If a config with the same name already exists, delete it first then recreate
cp_client = eval_client._control_plane_client.client

existing_configs = cp_client.list_online_evaluation_configs().get('onlineEvaluationConfigs', [])
existing = next((c for c in existing_configs if c.get('onlineEvaluationConfigName') == online_config_name), None)
if existing:
    cp_client.delete_online_evaluation_config(onlineEvaluationConfigId=existing['onlineEvaluationConfigId'])
    print(f"  Deleted existing config: {existing['onlineEvaluationConfigId']}")

print(f"Creating online evaluation configuration: '{online_config_name}'...")
print(f"  Using CDK eval execution role: {EVAL_EXECUTION_ROLE_ARN}")
online_config_response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name=online_config_name,
    sampling_rate=100,
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
    ],
    config_description=(
        "Continuous evaluation of Metadata Query Agent - "
        "monitors KB retrieval quality, SQL correctness, tool ordering, and goal completion."
    ),
    execution_role=EVAL_EXECUTION_ROLE_ARN,
)

online_config_id = online_config_response['onlineEvaluationConfigId']
print(f"\n✓ Online evaluation configuration created:")
print(f"  Config ID:   {online_config_id}")
print(f"  Config Name: {online_config_name}")
print(f"  Agent ID:    {agent_id}")
print(f"  Evaluators:  GoalSuccessRate, Correctness, ToolParameterAccuracy, ToolSelectionAccuracy")
print(f"  Sampling:    100%")

# Store config ID for future reference
online_config_id_stored = online_config_id
%store online_config_id_stored

Creating online evaluation config: semantic_layer_metadata_query_online_eval for agent: semantic_layer_metadata_query-itnnG8AGV6
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolParameterAccuracy', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: semantic_layer_metadata_query_online_eval for agent: semantic_layer_metadata_query-itnnG8AGV6


  Deleted existing config: semantic_layer_metadata_query_online_eval-hCm7vVHVms
Creating online evaluation configuration: 'semantic_layer_metadata_query_online_eval'...
  Using CDK eval execution role: arn:aws:iam::381492284087:role/semantic-layer-agentcore--EvalExecutionRole59F08324-ZX0EbSgXy1pm


✓ Online evaluation config created: semantic_layer_metadata_query_online_eval-YD96Te6g5o
✓ Online evaluation config created successfully
Config ID: semantic_layer_metadata_query_online_eval-YD96Te6g5o
Status: CREATING


✅ Online evaluation configuration created!


✓ Online evaluation configuration created:
  Config ID:   semantic_layer_metadata_query_online_eval-YD96Te6g5o
  Config Name: semantic_layer_metadata_query_online_eval
  Agent ID:    semantic_layer_metadata_query-itnnG8AGV6
  Evaluators:  GoalSuccessRate, Correctness, ToolParameterAccuracy, ToolSelectionAccuracy
  Sampling:    100%
Stored 'online_config_id_stored' (str)


### Confirm Configuration is Active

In [6]:
config_details = eval_client.get_online_config(config_id=online_config_id)
print(f"✓ Online evaluation configuration details:")
print(json.dumps(config_details, indent=2, default=str))

✓ Online evaluation configuration details:
{
  "ResponseMetadata": {
    "RequestId": "ab03cfa5-52e4-487d-8d9d-f11fd1033e9e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 23 Mar 2026 16:53:50 GMT",
      "content-type": "application/json",
      "content-length": "1289",
      "connection": "keep-alive",
      "x-amzn-requestid": "ab03cfa5-52e4-487d-8d9d-f11fd1033e9e",
      "x-amzn-remapped-x-amzn-requestid": "ab03cfa5-52e4-487d-8d9d-f11fd1033e9e",
      "x-amzn-remapped-content-length": "1289",
      "x-amzn-remapped-connection": "keep-alive",
      "x-cache": "Miss from cloudfront",
      "via": "1.1 25d9b5959eaa82bb18ee3f35e6bf34b4.cloudfront.net (CloudFront)",
      "x-amz-cf-id": "1hW-Z11lvlSGJegxBR0eYsMJb6gtasACpETpohKD2al3jirwoIz9mw==",
      "x-amz-apigw-id": "ar50vEOjIAMEhnA=",
      "x-amzn-trace-id": "Root=1-69c1701d-09d1988477431b1c351c8c33",
      "x-amz-cf-pop": "IAD12-P1",
      "x-amzn-remapped-date": "Mon, 23 Mar 2026 16:53:50 GMT"
    },
    "R

### Invoke Agent to Trigger Online Evaluation

Now that the online evaluation configuration is active, any agent invocation will automatically be evaluated. We invoke the agent with our test cases to generate traces that the online evaluator will process.

Each invocation uses a **unique `runtimeSessionId`** so each test case is evaluated as its own session.

In [7]:
agentcore_client = session.client('bedrock-agentcore', region_name=region)

invocation_results = []
print(f"Invoking Metadata Query Agent for {len(test_cases)} test case(s)...")
print(f"  Runtime ARN: {METADATA_QUERY_RUNTIME_ARN}")
print(f"  EVAL_ID:     {EVAL_ID}")
print(f"{'='*70}\n")

for tc in test_cases:
    session_id = str(uuid.uuid4())
    payload = json.dumps({'question': tc['query'], 'id': EVAL_ID}).encode('utf-8')

    print(f"[{tc['id']}] Session: {session_id}")
    print(f"       Query:   {tc['query']}")
    start = time.time()

    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=METADATA_QUERY_RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=payload,
        qualifier='DEFAULT',
    )

    # Read streaming response
    content = [chunk.decode('utf-8') for chunk in response.get('response', [])]
    response_text = ''.join(content)
    duration = time.time() - start

    try:
        response_data = json.loads(response_text)
    except json.JSONDecodeError:
        response_data = {'result': response_text}

    invocation_results.append({
        'test_id': tc['id'],
        'query': tc['query'],
        'session_id': session_id,
        'response': response_data,
        'duration': duration,
    })

    answer_preview = str(response_data.get('answer', response_data))[:120]
    print(f"       ✓ {duration:.1f}s — {answer_preview}...")
    print()

print(f"{'='*70}")
print(f"✓ All {len(test_cases)} test case(s) invoked")
print(f"  Traces are now flowing into AgentCore Observability")
print(f"  Online evaluation will process them automatically")

invoked_session_ids = [r['session_id'] for r in invocation_results]
print(f"\n  Session IDs:")
for r in invocation_results:
    print(f"    [{r['test_id']}] {r['session_id']}")

Invoking Metadata Query Agent for 1 test case(s)...
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:381492284087:runtime/semantic_layer_metadata_query-itnnG8AGV6
  EVAL_ID:     semantic-rag-single_table_basic-8f7bcf77

[test1] Session: 4f6593b3-3c8d-481e-a9cc-823de923edbf
       Query:   How many unique admincodes exist?
       ✓ 17.1s — There are **10 unique admin codes** in the `admin_codes` table, as identified by the distinct values of the `admin_code_...

✓ All 1 test case(s) invoked
  Traces are now flowing into AgentCore Observability
  Online evaluation will process them automatically

  Session IDs:
    [test1] 4f6593b3-3c8d-481e-a9cc-823de923edbf


### Wait for OTEL Spans to Be Indexed

AgentCore Observability uses CloudWatch Logs Insights to index OTEL spans. Online evaluation runs automatically once spans are indexed — typically within 90 seconds of invocation.

In [8]:
import time as _time

_logs = session.client('logs', region_name=region)
_fixed_wait = 90  # seconds — empirically safe for full span indexing

print(f"Sleeping {_fixed_wait}s for CWL Insights to finish indexing all spans...")
for _i in range(_fixed_wait // 10):
    _time.sleep(10)
    print(f"  [{(_i+1)*10}s] ...")

# Confirm at least one invoke_agent span is present for the first session
_first_session = invoked_session_ids[0] if invoked_session_ids else None
_confirmed = False

if _first_session:
    print(f"Confirming spans indexed for session {_first_session} ...")
    for _attempt in range(4):
        try:
            _resp = _logs.start_query(
                logGroupName='aws/spans',
                startTime=int(_time.time()) - 600,
                endTime=int(_time.time()),
                queryString=(
                    f"fields spanId | filter attributes.session.id = '{_first_session}' "
                    f"| filter name = 'invoke_agent Strands Agents' "
                    f"| parse resource.attributes.cloud.resource_id \"runtime/*/\" as parsedAgentId "
                    f"| filter parsedAgentId = '{agent_id}' | limit 1"
                ),
            )
            for _ in range(20):
                _time.sleep(1)
                _r = _logs.get_query_results(queryId=_resp['queryId'])
                if _r['status'] == 'Complete':
                    if _r.get('results'):
                        _confirmed = True
                    break
        except Exception as _e:
            print(f"  confirm error: {_e}")
        if _confirmed:
            break
        print(f"  invoke_agent not yet visible, waiting 15s more...")
        _time.sleep(15)

if _confirmed:
    print(f"✓ Spans indexed — online evaluation is now processing the sessions")
else:
    print(f"⚠️  Span confirmation inconclusive — online evaluation may still be processing.")
    print(f"   Check AgentCore Observability dashboard in a few minutes.")

Sleeping 90s for CWL Insights to finish indexing all spans...
  [10s] ...
  [20s] ...
  [30s] ...
  [40s] ...
  [50s] ...
  [60s] ...
  [70s] ...
  [80s] ...
  [90s] ...
Confirming spans indexed for session 4f6593b3-3c8d-481e-a9cc-823de923edbf ...
✓ Spans indexed — online evaluation is now processing the sessions


### Invocation Summary

In [9]:
summary_rows = []
for r in invocation_results:
    resp = r['response']
    answer = resp.get('answer', resp.get('result', str(resp)))[:80] + '...'
    summary_rows.append({
        'Test ID': r['test_id'],
        'Query': r['query'][:50] + ('...' if len(r['query']) > 50 else ''),
        'Duration (s)': f"{r['duration']:.1f}",
        'Session ID (short)': r['session_id'][:8] + '...',
        'Answer Preview': answer,
    })

df_summary = pd.DataFrame(summary_rows)
print(f"Agent: {agent_id}")
print(f"EVAL_ID: {EVAL_ID}")
print(f"Online Config ID: {online_config_id}")
print(f"\n{'='*70}")
display(df_summary)

Agent: semantic_layer_metadata_query-itnnG8AGV6
EVAL_ID: semantic-rag-single_table_basic-8f7bcf77
Online Config ID: semantic_layer_metadata_query_online_eval-YD96Te6g5o



,Test ID,Query,Duration (s),Session ID (short),Answer Preview
0,test1,How many unique admincodes exist?,17.1,4f6593b3...,There are **10 unique admin codes** in the `ad...


### Viewing Results in AgentCore Observability

Online evaluation results appear automatically in the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) once spans are processed.

Navigate to your agent's `DEFAULT` endpoint to see:
- **Session-level** `GoalSuccessRate` scores
- **Trace-level** `Correctness` scores per invocation
- **Span-level** `ToolParameterAccuracy` and `ToolSelectionAccuracy` scores per tool call

> **Note:** Evaluation results may take a few minutes to appear after invocation. If the dashboard is empty, wait 2-3 minutes and refresh.

### Managing the Online Evaluation Configuration

You can list, retrieve, update, or delete the online evaluation configuration using the eval client.

In [10]:
# Retrieve the current online evaluation config
current_config = eval_client.get_online_config(config_id=online_config_id)
print(f"✓ Current online evaluation configuration:")
print(json.dumps(current_config, indent=2, default=str))

# To delete the config (uncomment when no longer needed):
# cp_client.delete_online_evaluation_config(onlineEvaluationConfigId=online_config_id)
# print(f"✓ Deleted online evaluation configuration: {online_config_id}")

# To list all configs for this account:
# all_configs = cp_client.list_online_evaluation_configs().get('onlineEvaluationConfigs', [])
# for c in all_configs:
#     print(c['onlineEvaluationConfigId'], c['onlineEvaluationConfigName'], c['executionStatus'])

✓ Current online evaluation configuration:
{
  "ResponseMetadata": {
    "RequestId": "bc1b2fec-a178-447d-8baf-ac7dc4dfc28d",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 23 Mar 2026 16:55:38 GMT",
      "content-type": "application/json",
      "content-length": "1287",
      "connection": "keep-alive",
      "x-amzn-requestid": "bc1b2fec-a178-447d-8baf-ac7dc4dfc28d",
      "x-amzn-remapped-x-amzn-requestid": "bc1b2fec-a178-447d-8baf-ac7dc4dfc28d",
      "x-amzn-remapped-content-length": "1287",
      "x-amzn-remapped-connection": "keep-alive",
      "x-cache": "Miss from cloudfront",
      "via": "1.1 76f3fedc86826a7b266250e33ee41082.cloudfront.net (CloudFront)",
      "x-amz-cf-id": "HsotIRTkSqfAuHvqyCvtOwdUPWKStUejh2z7RclAaSDduomkgYASUA==",
      "x-amz-apigw-id": "ar6FvHKXIAMEXfQ=",
      "x-amzn-trace-id": "Root=1-69c1708a-5b733e1a74de3b793e4a233c",
      "x-amz-cf-pop": "IAD12-P1",
      "x-amzn-remapped-date": "Mon, 23 Mar 2026 16:55:38 GMT"
    },
    "R

## Summary

This notebook set up **online (continuous) evaluation** of the Metadata Query Agent on AgentCore Runtime:

1. **Restored** `metadata_id` from notebook 1 and `ondemand_agent_id` from notebook 3 via `%store`
2. **Looked up** the AgentCore Runtime ARN from CloudFormation stack outputs
3. **Created** an online evaluation configuration with four built-in evaluators:
   - `GoalSuccessRate` (session-level)
   - `Correctness` (trace-level)
   - `ToolParameterAccuracy` (span-level)
   - `ToolSelectionAccuracy` (span-level)
4. **Invoked** the agent with test queries — each in its own session — to generate new traces
5. **Confirmed** spans are indexed in CloudWatch Logs Insights

### Notebook Progression

| Notebook | Agent | Approach | Use case |
|:---------|:------|:---------|:---------|
| 1 — Metadata Agent Strands eval | Metadata Generation | Local Strands, in-memory telemetry | Development iteration |
| 2 — Metadata Query Agent Strands eval | Metadata Query | Local Strands, in-memory telemetry | Development iteration |
| 3 — AgentCore on-demand eval | Metadata Query | AgentCore Runtime + CWL, on-demand | Pre-production validation |
| **4 — AgentCore online eval** | **Metadata Query** | **AgentCore Runtime + CWL, continuous** | **Production monitoring** |

### Next Steps
- Monitor the [AgentCore Observability dashboard](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) for live evaluation scores
- Expand the evaluation dataset with multi-table joins and edge cases
- Adjust `sampling_rate` in production to balance cost vs. coverage
- Add custom evaluators for domain-specific quality criteria (e.g. financial terminology accuracy)